In [1]:
# 0. 导库
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
import yaml

In [4]:
data_path = '../data/raw/test.pkl'

In [5]:
raw_data = joblib.load(data_path)

In [6]:
raw_data.keys()

dict_keys(['datasets', 'label_mapping', 'fs'])

In [7]:
# 针对于uECoG数据
def reconstruct_datasets(raw_data):
    datas_list = raw_data['datasets']
    fs = raw_data['fs']
    label_mapping = {0:"non_task",1:"task"}

    rest_data = datas_list[0]
    task_data_list = []
    for i in range(1,len(datas_list)):
        task_data_list.append(datas_list[i])
    task_data = np.concatenate(task_data_list,axis=0)

    rest_data = rest_data[:,:128,:]
    task_data = task_data[:,:128,:]
    rest_labels = np.ones(rest_data.shape[0]) * 0
    task_labels = np.ones(task_data.shape[0]) * 1

    datasets = {0: (rest_data,rest_labels),1:(task_data,task_labels)}
    dataset = {"datasets":datasets,"label_mapping":label_mapping,"fs":fs}

    joblib.dump(dataset,os.path.join('../data/reconstruct_datasets',data_path.split("/")[3]))


In [8]:
reconstruct_datasets(raw_data)

In [2]:
# 1. 数据预处理
from preprocess.preprocessor import Preprocessor

In [3]:
dataset = joblib.load("../data/reconstruct_datasets/test.pkl")
datasets = dataset['datasets']
label_mapping = dataset['label_mapping']
fs = dataset['fs']

In [12]:
datasets.keys()

dict_keys([0, 1])

In [17]:
datasets[0][0].shape,datasets[0][1].shape

((100, 128, 2400), (100,))

In [4]:
preprocessor = Preprocessor(dataset)
preprocessed_data = preprocessor.preprocess([50,100,150],1,200,2,4)

Preprocess time: 3.04s


In [8]:
preprocessed_data[0].shape

(100, 128, 2400)

In [10]:
preprocessed_data[1].shape

(380, 128, 2400)

In [11]:
preprocessed_datasets = {0:preprocessed_data[0],1:preprocessed_data[1]}

In [9]:
from featureExtract.feature_extract import FeatureExtractor

In [13]:
fe = FeatureExtractor(fs)
X,y = fe.extract_features_and_labels(preprocessed_datasets)

In [17]:
X.shape,y.shape

((480, 1536), (480,))